# Trading Game #2 — Optimized Pairs Trading
### Student: Margaux | Commodity: Aluminium (G5)
### Course: Commodities Markets & Models — ESILV | March 2026

**Reference:** Palazzi, R.B. (2025). Trading Games: Beating Passive Strategies in the Bullish Crypto Market. *Journal of Futures Markets*, 45(11), 1911–1933.  
**GitHub:** https://github.com/rafaelpalazzi/trading-games-crypto

In [ ]:
!pip install yfinance statsmodels -q

## 1. Data Collection & Cleaning

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

BENCHMARK = 'ALI=F'

ASSETS = [
    'PICK', 'XME', 'XLB',
    'AA', 'CENX', 'KALU', 'CSTM',
    'RIO', 'NHYDY', 'ACH',
    'BHP', 'HINDALCO.NS', 'S32.AX',
    'FCX', 'NEM',
]

ALL_TICKERS = [BENCHMARK] + ASSETS
START_DATE  = '2018-01-01'
END_DATE    = '2024-12-31'

print(f'Downloading {len(ALL_TICKERS)} tickers...')
raw    = yf.download(ALL_TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=True)
prices = raw['Close'].copy()
print(f'Raw: {prices.shape[0]} days x {prices.shape[1]} assets')

In [ ]:
nan_pct = prices.isna().mean()
keep    = nan_pct[nan_pct <= 0.20].index.tolist()
dropped = [t for t in ALL_TICKERS if t not in keep]
if dropped:
    print(f'Dropped: {dropped}')

prices = prices[keep].ffill(limit=3).dropna()
print(f'After cleaning: {prices.shape[0]} days x {prices.shape[1]} assets')
print(f'Period: {prices.index[0].date()} to {prices.index[-1].date()}')

log_prices  = np.log(prices)
log_returns = log_prices.diff().dropna()

prices.to_csv('prices_clean.csv')
log_prices.to_csv('log_prices.csv')
log_returns.to_csv('log_returns.csv')
print('Saved: prices_clean.csv, log_prices.csv, log_returns.csv')

## 2. Descriptive Statistics

In [ ]:
ann_ret = log_returns.mean() * 252
ann_vol = log_returns.std()  * np.sqrt(252)
sharpe  = ann_ret / ann_vol
cum     = (1 + log_returns).cumprod()
mdd     = ((cum - cum.cummax()) / cum.cummax()).min()
calmar  = ann_ret / mdd.abs()

stats = pd.DataFrame({
    'Ann. Return (%)':     (ann_ret * 100).round(2),
    'Ann. Volatility (%)': (ann_vol * 100).round(2),
    'Sharpe Ratio':         sharpe.round(3),
    'Max Drawdown (%)':    (mdd * 100).round(2),
    'Calmar Ratio':         calmar.round(3),
    'Skewness':             log_returns.skew().round(3),
    'Excess Kurtosis':      log_returns.kurt().round(3),
}).sort_values('Ann. Return (%)', ascending=False)

print('Descriptive Statistics:')
stats

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 11))
fig.suptitle('Aluminium Universe — Data Overview', fontsize=13, fontweight='bold')

norm = prices / prices.iloc[0] * 100
for col in norm.columns:
    if col == BENCHMARK:
        axes[0].plot(norm.index, norm[col], color='crimson', lw=2.5, zorder=5, label=f'{BENCHMARK} (benchmark)')
    else:
        axes[0].plot(norm.index, norm[col], lw=0.8, alpha=0.4, color='steelblue')
axes[0].plot([], [], color='steelblue', lw=0.8, alpha=0.4, label=f'Related assets (n={len(keep)-1})')
axes[0].set_title('Normalised Prices (base 100)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.25)

corr = log_returns.corr()
im   = axes[1].imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(keep)))
axes[1].set_yticks(range(len(keep)))
axes[1].set_xticklabels(keep, rotation=45, ha='right', fontsize=7)
axes[1].set_yticklabels(keep, fontsize=7)
axes[1].set_title('Return Correlation Matrix')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
for i in range(len(keep)):
    for j in range(len(keep)):
        v = corr.values[i, j]
        axes[1].text(j, i, f'{v:.2f}', ha='center', va='center',
                     fontsize=5, color='white' if abs(v) > 0.65 else 'black')
plt.tight_layout()
plt.savefig('step1_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Cointegration Testing & Pair Selection

In [ ]:
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from itertools import combinations

print('Running Engle-Granger cointegration tests on all pairs...')

results = []
for t1, t2 in combinations(keep, 2):
    try:
        score, pval, _ = coint(log_prices[t1], log_prices[t2])
        results.append({
            'Asset 1': t1, 'Asset 2': t2,
            't-stat': round(score, 4),
            'p-value': round(pval, 4),
            'Cointegrated': pval < 0.05,
            'With benchmark': (t1 == BENCHMARK or t2 == BENCHMARK),
        })
    except:
        pass

coint_df = pd.DataFrame(results).sort_values('p-value')
print(f'Tested {len(results)} pairs — {coint_df["Cointegrated"].sum()} cointegrated at 5%')
print('\nTop 15:')
coint_df.head(15)

In [ ]:
ali_pairs = coint_df[coint_df['With benchmark']]
print('Pairs involving ALI=F:')
display(ali_pairs)

best  = ali_pairs.iloc[0]
A1    = best['Asset 1']
A2    = best['Asset 2']
print(f'\nSelected: ({A1}, {A2}) | p-value = {best["p-value"]}')

y     = log_prices[A1].values
x     = add_constant(log_prices[A2].values)
model = OLS(y, x).fit()
BETA  = model.params[1]
ALPHA = model.params[0]

spread = log_prices[A1] - BETA * log_prices[A2] - ALPHA

adf_stat, adf_pval = adfuller(spread)[:2]
print(f'Spread model: log({A1}) = {ALPHA:.4f} + {BETA:.4f} * log({A2})')
print(f'R² = {model.rsquared:.4f}')
print(f'ADF test: stat={adf_stat:.4f}, p={adf_pval:.4f} -> {"stationary" if adf_pval < 0.05 else "NOT stationary"}')

coint_df.to_csv('cointegration_results.csv', index=False)

In [ ]:
LOOKBACK  = 60
roll_mean = spread.rolling(LOOKBACK).mean()
roll_std  = spread.rolling(LOOKBACK).std()
zscore_60 = (spread - roll_mean) / roll_std

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle(f'Cointegration Analysis: {A1} / {A2}', fontsize=13, fontweight='bold')

(log_prices[[A1, A2]] - log_prices[[A1, A2]].iloc[0]).plot(ax=axes[0])
axes[0].set_title('Log-Prices (demeaned)')
axes[0].grid(alpha=0.25)

spread.plot(ax=axes[1], color='darkorange', lw=1.0)
roll_mean.plot(ax=axes[1], color='black', lw=1.2, linestyle='--', label='Rolling mean')
axes[1].fill_between(spread.index, roll_mean - roll_std, roll_mean + roll_std, alpha=0.15, color='grey')
axes[1].set_title(f'Spread = log({A1}) - {BETA:.4f}*log({A2})')
axes[1].legend()
axes[1].grid(alpha=0.25)

zscore_60.plot(ax=axes[2], color='steelblue', lw=0.9)
axes[2].axhline( 1, color='red',   linestyle='--', lw=1.2)
axes[2].axhline(-1, color='green', linestyle='--', lw=1.2)
axes[2].axhline( 0, color='black', lw=0.8)
axes[2].set_title('Z-Score (lookback=60d)')
axes[2].grid(alpha=0.25)

plt.tight_layout()
plt.savefig('step2_cointegration.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Pairs Trading Strategy

In [ ]:
import itertools

def zscore(spread, lb):
    mu  = spread.rolling(lb).mean()
    sig = spread.rolling(lb).std()
    return (spread - mu) / sig

def sharpe(r, freq=252):
    r = r.dropna()
    return r.mean() / r.std() * np.sqrt(freq) if r.std() > 0 else 0.0

def run_backtest(spread, log_prices, a1, a2, beta,
                 lb, threshold, tc=0.002,
                 stop_pct=0.025, min_hold=5,
                 vol_lb=30, vol_mult=1.5):
    z       = zscore(spread, lb)
    vol     = spread.rolling(vol_lb).std()
    vol_avg = vol.rolling(vol_lb).mean()
    n       = len(spread)
    sigs    = np.zeros(n)
    rets    = np.zeros(n)
    pos, hold, entry = 0, 0, np.nan

    for t in range(lb, n):
        if np.isnan(z.iloc[t]) or np.isnan(vol.iloc[t]):
            continue
        high_vol = vol.iloc[t] > vol_mult * vol_avg.iloc[t]
        stop_hit = pos != 0 and not np.isnan(entry) and \
                   (spread.iloc[t] - entry) * pos < -stop_pct * abs(entry + 1e-8)
        if pos != 0:
            hold += 1
        can_close = (hold >= min_hold) or stop_hit
        new_sig = 0
        if not high_vol:
            if z.iloc[t] <= -threshold:
                new_sig = 1
            elif z.iloc[t] >= threshold:
                new_sig = -1
        if pos == 0 and new_sig != 0:
            pos, sigs[t], hold, entry, rets[t] = new_sig, new_sig, 0, spread.iloc[t], -tc
        elif pos != 0 and can_close and (new_sig != pos or stop_hit):
            rets[t] = -tc
            pos, sigs[t], hold = new_sig, new_sig, 0
            entry = spread.iloc[t] if pos != 0 else np.nan
            if new_sig != 0:
                rets[t] -= tc
        elif pos != 0:
            sigs[t] = pos
            rets[t] = pos * (log_prices[a1].diff().iloc[t] - beta * log_prices[a2].diff().iloc[t])

    return pd.Series(rets, index=spread.index), pd.Series(sigs, index=spread.index)

## 5. Parameter Optimisation (Grid Search)

In [ ]:
split    = int(len(spread) * 0.75)
sp_train = spread.iloc[:split]
lp_train = log_prices.iloc[:split]

print(f'Train: {sp_train.index[0].date()} to {sp_train.index[-1].date()} ({split} days)')
print(f'Test:  {spread.index[split].date()} to {spread.index[-1].date()} ({len(spread)-split} days)')

LB_GRID  = [20, 30, 40, 60, 90, 120]
TH_GRID  = [0.5, 0.7, 1.0, 1.2, 1.5]

best_sr, best_lb, best_th = -np.inf, 20, 0.7
grid_res = []

for lb, th in itertools.product(LB_GRID, TH_GRID):
    r, _ = run_backtest(sp_train, lp_train, A1, A2, BETA, lb, th)
    sr   = sharpe(r)
    grid_res.append({'lookback': lb, 'threshold': th, 'sharpe': round(sr, 4)})
    if sr > best_sr:
        best_sr, best_lb, best_th = sr, lb, th

grid_df = pd.DataFrame(grid_res).sort_values('sharpe', ascending=False)
print(f'Best params: lookback={best_lb}, threshold={best_th}, in-sample sharpe={best_sr:.3f}')
grid_df.head(10)

## 6. Out-of-Sample Backtest & Results

In [ ]:
oos_rets, oos_sigs = run_backtest(spread, log_prices, A1, A2, BETA, best_lb, best_th)
oos_rets = oos_rets.iloc[split:]
oos_sigs = oos_sigs.iloc[split:]
bh_rets  = log_prices[BENCHMARK].diff().iloc[split:].dropna()

def perf(returns, name, n_trades=None):
    r   = returns.dropna()
    cum = (1 + r).cumprod()
    ann = r.mean() * 252
    mdd = ((cum - cum.cummax()) / cum.cummax()).min()
    return {
        'Strategy':            name,
        'Ann. Return (%)':     round(ann * 100, 2),
        'Ann. Volatility (%)': round(r.std() * np.sqrt(252) * 100, 2),
        'Sharpe Ratio':        round(sharpe(r), 3),
        'Max Drawdown (%)':    round(mdd * 100, 2),
        'Calmar Ratio':        round(ann / abs(mdd), 3) if mdd != 0 else np.nan,
        'Total Return (%)':    round((cum.iloc[-1] - 1) * 100, 2),
        '# Trades':            n_trades if n_trades is not None else 'N/A',
    }

n_trades = int((oos_sigs.diff().abs() > 0).sum())
perf_df  = pd.DataFrame([
    perf(oos_rets, 'Pairs Trading', n_trades),
    perf(bh_rets,  f'Buy-and-Hold {BENCHMARK}'),
]).set_index('Strategy')

print('Out-of-sample performance:')
perf_df

In [ ]:
oos_cum = (1 + oos_rets.fillna(0)).cumprod()
bh_cum  = (1 + bh_rets.fillna(0)).cumprod()
z_oos   = zscore(spread, best_lb).iloc[split:]

fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)
fig.suptitle(f'TG2 — Out-of-Sample Results: {A1}/{A2} | lb={best_lb} th={best_th}',
             fontsize=13, fontweight='bold')

oos_cum.plot(ax=axes[0], color='steelblue', lw=2, label='Pairs Trading')
bh_cum.plot( ax=axes[0], color='crimson',   lw=2, linestyle='--', label=f'Buy-and-Hold {BENCHMARK}')
axes[0].set_title('Cumulative Returns')
axes[0].set_ylabel('Portfolio value')
axes[0].legend()
axes[0].grid(alpha=0.25)

z_oos.plot(ax=axes[1], color='darkorange', lw=0.9, alpha=0.8)
axes[1].axhline( best_th, color='red',   linestyle='--', lw=1.2, label=f'+{best_th}')
axes[1].axhline(-best_th, color='green', linestyle='--', lw=1.2, label=f'-{best_th}')
axes[1].axhline(0, color='black', lw=0.7)
axes[1].fill_between(z_oos.index, z_oos, 0, where=(oos_sigs==1),  alpha=0.2, color='green')
axes[1].fill_between(z_oos.index, z_oos, 0, where=(oos_sigs==-1), alpha=0.2, color='red')
axes[1].set_title('Z-Score and signals')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.25)

dd_strat = (oos_cum / oos_cum.cummax() - 1) * 100
dd_bh    = (bh_cum  / bh_cum.cummax()  - 1) * 100
dd_strat.plot(ax=axes[2], color='steelblue', lw=1, label='Strategy')
dd_bh.plot(   ax=axes[2], color='crimson',   lw=1, linestyle='--', label='Buy-and-Hold')
axes[2].set_title('Drawdown (%)')
axes[2].legend()
axes[2].grid(alpha=0.25)

plt.tight_layout()
plt.savefig('step3_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

perf_df.to_csv('performance_summary.csv')
grid_df.to_csv('grid_search_results.csv', index=False)
print('Saved all outputs.')